In [ ]:
# --- Environment Setup (do not edit) ---
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401

        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    print("\u26a0\ufe0f  WARNING: Could not detect platform. Falling back to .env (local).")
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    print(
        f"\u26a0\ufe0f  WARNING: Detected /workspace but VAST_CONTAINERLABEL is not set. Assuming Vast.ai."
    )
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    print(f"\u26a0\ufe0f  WARNING: Expected env file not found: {_env_file}")
    if _fallback.exists():
        print(f"   Falling back to: {_fallback}")
        _env_file = _fallback
    else:
        raise FileNotFoundError(
            f"No .env file found. Copy the correct example file:\n"
            f"  cp .env.{PLATFORM or 'mac'}.example .env.{PLATFORM or 'mac'}"
        )

load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"\u2705 Platform : {PLATFORM or 'unknown (local fallback)'}")
print(f"\u2705 Env file : {_env_file}")
print(f"\u2705 DATA_ROOT: {DATA_ROOT}")
print(f"{'\u2705' if DATA_ROOT.exists() else '\u274c'} Path exists: {DATA_ROOT.exists()}")
# ---------------------------------------------------------------------------

# ViFactCheck Other PLMs Training (Stage 3.9b)

Fine-tunes **XLM-R**, **mBERT**, and **ViBERT** on ViFactCheck, following the original paper's PLM experiments.

Paper §4.1 evaluates four PLMs: mBERT, XLM-R, PhoBERT, ViBERT.
This notebook trains the three non-PhoBERT models using the same original architecture and hyperparameters.

- **Models**: `xlm-roberta-base`, `bert-base-multilingual-cased`, `vinai/vibert-base`
- **Architecture**: Pooler output → Dropout(0.3) → Linear (same as original)
- **Input**: `Statement + Context` (full context mode, matching original `plm_training.py`)
- **Loss**: Plain `CrossEntropyLoss()`
- **Optimizer**: `AdamW(lr=5e-6)`
- **Training**: Fixed 10 epochs, no early stopping
- **Classification**: 3-class (Supported=0, Refuted=1, NEI=2)

```
Output: training/checkpoints_vifactcheck_plms/<model_name>/<run>/best_model.pth
```

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv

    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

# Models to train (paper §4.1: mBERT, XLM-R, ViBERT — PhoBERT is in separate notebooks)
MODELS_TO_TRAIN = [
    "xlm-roberta-base",
    "bert-base-multilingual-cased",
    "vinai/vibert-base",
]

CONFIG = {
    "paths": {
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck_plms",
    },
    "data": {
        "hf_dataset": "tranthaihoa/vifactcheck",
        "text_field": "Statement",
        "context_field": "Context",  # Full context mode (original plm_training.py)
        "label_field": "labels",
        "max_length": 256,
        "num_classes": 3,
    },
    "model": {
        "dropout": 0.3,
        "num_classes": 3,
    },
    "training": {
        "batch_size": 16,
        "epochs": 10,
        "lr": 0.5e-5,
        "seed": 28,
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 2,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Checkpoint:   {CONFIG['paths']['checkpoint_root']}")
print(f"Models:       {MODELS_TO_TRAIN}")

In [ ]:
# --- DEPENDENCY PREFLIGHT ---
import importlib, sys

_required = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}

_missing = []
for mod, pkg in _required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)

if _missing:
    print(f"Missing packages: {_missing}")
    raise RuntimeError(f"Missing packages: {_missing}.")
else:
    print("All dependencies satisfied.")

In [ ]:
# --- IMPORTS AND SETUP ---
import os, gc, json, random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from transformers import AutoTokenizer, AutoModel

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)

print(f"PyTorch      : {torch.__version__}")
import transformers

print(f"Transformers : {transformers.__version__}")


def select_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("Device: mps (Apple Silicon). For full training, prefer a CUDA GPU.")
    else:
        dev = torch.device("cpu")
        print("Device: cpu. Use smoke_test=True for local validation.")
    return dev


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = select_device()
seed_everything(CONFIG["training"]["seed"])
print(f"Seed: {CONFIG['training']['seed']}")

## Step 1: Load ViFactCheck Dataset

Loads once, reuses across all model runs. `Statement + Context` sentence pairs, 3-class labels.

In [ ]:
from datasets import load_dataset

hf_name = CONFIG["data"]["hf_dataset"]
text_field = CONFIG["data"]["text_field"]
ctx_field = CONFIG["data"]["context_field"]
label_field = CONFIG["data"]["label_field"]

print(f"Loading '{hf_name}' from HuggingFace...")
raw = load_dataset(hf_name)

available = list(raw.keys())
print(f"  Available splits: {available}")

raw_examples = {}
for hf_split in available:
    ds = raw[hf_split]
    examples = []
    for item in ds:
        stmt = str(item.get(text_field) or "").strip()
        ctx = str(item.get(ctx_field) or "").strip()
        label = int(item.get(label_field))
        if not stmt:
            continue
        examples.append({"text": stmt, "text_b": ctx, "label": label})
    nc = CONFIG["data"]["num_classes"]
    hist = [sum(1 for e in examples if e["label"] == i) for i in range(nc)]
    print(f"  {hf_split:5s}: {len(examples):5d} examples | label dist {hist}")
    raw_examples[hf_split] = examples

print(f"\nData loaded: { {k: len(v) for k, v in raw_examples.items()} }")

## Step 2: Model Architecture and Training Functions

Same `PLMClassifier` architecture as original ViFactCheck code: pooler → dropout(0.3) → linear.
Works with any BERT/RoBERTa-style model that has a pooler output.

In [ ]:
class SentencePairDataset(Dataset):
    def __init__(self, sentence_pairs, labels, tokenizer, max_length):
        self.sentence_pairs = sentence_pairs
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentence_pairs)

    def __getitem__(self, idx):
        sentence1, sentence2 = self.sentence_pairs[idx]
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            sentence1,
            text_pair=sentence2,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding="max_length",
            return_attention_mask=True,
            return_tensors="pt",
            truncation=True,
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long),
        }


class PLMClassifier(nn.Module):
    """Generic PLM classifier matching original ViFactCheck architecture."""

    def __init__(self, backbone, num_classes):
        super(PLMClassifier, self).__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.3)
        self.linear = nn.Linear(self.backbone.config.hidden_size, num_classes)
        self.hidden_size = self.backbone.config.hidden_size

    def forward(self, input_ids, attention_mask):
        _, pooled_output = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False,
        )
        dropout_output = self.dropout(pooled_output)
        logits = self.linear(dropout_output)
        return logits


def compute_metrics(y_true, y_pred, nc):
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    return {"accuracy": round(acc, 4), "macro_f1": round(macro_f1, 4),
            "per_f1": [round(float(f), 4) for f in per_f1]}


print("Model and dataset classes defined.")

## Step 3: Train All Models

Loops over `MODELS_TO_TRAIN`, training each with the same original hyperparameters.
Saves best checkpoint + tokenizer + manifest per model.

In [ ]:
all_results = {}
_max_len = CONFIG["data"]["max_length"]
_bs = CONFIG["training"]["batch_size"]
_epochs = CONFIG["training"]["epochs"]
_lr = CONFIG["training"]["lr"]
_nc = CONFIG["model"]["num_classes"]
_smoke = CONFIG["safety"]["smoke_test"]
_smoke_n = CONFIG["safety"]["smoke_batches"]

for model_idx, backbone_name in enumerate(MODELS_TO_TRAIN):
    model_tag = backbone_name.split("/")[-1]
    print(f"\n{'='*70}")
    print(f"Model {model_idx+1}/{len(MODELS_TO_TRAIN)}: {backbone_name}")
    print(f"{'='*70}")

    seed_everything(CONFIG["training"]["seed"])

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(backbone_name)

    # Build datasets
    datasets_map = {}
    for split, exs in raw_examples.items():
        pairs = [(e["text"], e["text_b"]) for e in exs]
        labels = [e["label"] for e in exs]
        datasets_map[split] = SentencePairDataset(pairs, labels, tokenizer, _max_len)

    loaders = {
        "train": DataLoader(datasets_map["train"], batch_size=_bs, shuffle=True, num_workers=0),
        "dev": DataLoader(datasets_map.get("dev", datasets_map["train"]), batch_size=_bs, shuffle=False, num_workers=0),
        "test": DataLoader(datasets_map.get("test", datasets_map["train"]), batch_size=_bs, shuffle=False, num_workers=0),
    }

    if _smoke:
        from itertools import islice
        class _SmokeLoader:
            def __init__(self, loader, n):
                self._l = loader
                self._n = n
            def __iter__(self):
                return islice(self._l, self._n)
            def __len__(self):
                return min(self._n, len(self._l))
        loaders = {k: _SmokeLoader(v, _smoke_n) for k, v in loaders.items()}

    # Build model
    backbone = AutoModel.from_pretrained(backbone_name)
    model = PLMClassifier(backbone, num_classes=_nc).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}  Hidden: {model.hidden_size}")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=_lr)

    # Run dir
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = CONFIG["paths"]["checkpoint_root"] / model_tag / f"run_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = run_dir / "best_model.pth"

    best_val_f1 = -1.0
    best_epoch = -1
    history = []

    for epoch in range(_epochs):
        model.train()
        train_loss = 0.0
        n_batches = 0
        pbar = tqdm(loaders["train"], desc=f"  Epoch {epoch+1}/{_epochs} [train]", leave=False)
        for batch in pbar:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        mean_train_loss = train_loss / max(1, n_batches)

        model.eval()
        predictions, true_labels = [], []
        for batch in tqdm(loaders["dev"], desc=f"  Epoch {epoch+1}/{_epochs} [dev]", leave=False):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            with torch.no_grad():
                outputs = model(input_ids, attention_mask)
                _, predicted = torch.max(outputs, 1)
                predictions.extend(predicted.cpu().numpy().tolist())
                true_labels.extend(labels.cpu().numpy().tolist())

        dev_metrics = compute_metrics(true_labels, predictions, _nc)
        print(f"  Epoch {epoch+1}/{_epochs}  loss={mean_train_loss:.4f}  "
              f"dev_acc={dev_metrics['accuracy']:.4f}  dev_f1={dev_metrics['macro_f1']:.4f}")

        history.append({
            "epoch": epoch + 1,
            "train_loss": round(mean_train_loss, 4),
            "dev_accuracy": dev_metrics["accuracy"],
            "dev_macro_f1": dev_metrics["macro_f1"],
        })

        if dev_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = dev_metrics["macro_f1"]
            best_epoch = epoch + 1
            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": best_epoch,
                "dev_metrics": dev_metrics,
                "backbone": backbone_name,
                "hidden_size": model.hidden_size,
                "num_classes": _nc,
            }, best_ckpt_path)

    # Test evaluation
    _ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(_ckpt["model_state_dict"])
    model.eval()
    predictions, true_labels = [], []
    for batch in tqdm(loaders["test"], desc="  Test", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy().tolist())
            true_labels.extend(labels.cpu().numpy().tolist())

    test_metrics = compute_metrics(true_labels, predictions, _nc)
    print(f"\n  TEST: acc={test_metrics['accuracy']:.4f}  macro_f1={test_metrics['macro_f1']:.4f}")
    print(classification_report(true_labels, predictions, digits=4))

    # Save tokenizer + manifest
    tokenizer_dir = run_dir / "tokenizer"
    tokenizer.save_pretrained(str(tokenizer_dir))

    manifest = {
        "best_checkpoint_path": str(best_ckpt_path),
        "tokenizer_path": str(tokenizer_dir),
        "best_epoch": best_epoch,
        "best_dev_macro_f1": best_val_f1,
        "test_metrics": test_metrics,
        "backbone": backbone_name,
        "hidden_size": model.hidden_size,
        "num_classes": _nc,
        "input_format": "Statement + Context",
        "architecture": "pooler_output -> Dropout(0.3) -> Linear",
        "training_setup": {
            "loss": "CrossEntropyLoss (plain)",
            "optimizer": "AdamW",
            "lr": _lr,
            "epochs": _epochs,
            "batch_size": _bs,
            "max_length": _max_len,
        },
        "data_source": "HuggingFace tranthaihoa/vifactcheck",
        "label_scheme": "3-class: Supported(0), Refuted(1), NEI(2)",
    }
    with open(run_dir / "checkpoint_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    all_results[model_tag] = {
        "backbone": backbone_name,
        "best_epoch": best_epoch,
        "best_dev_f1": best_val_f1,
        "test_metrics": test_metrics,
        "run_dir": str(run_dir),
    }

    # Cleanup
    del model, backbone, tokenizer, optimizer, criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print("All models trained.")
print(f"{'='*70}")

## Step 4: Summary Comparison Table

In [ ]:
if not all_results:
    print("No results to summarize.")
else:
    rows = []
    for tag, res in all_results.items():
        rows.append({
            "Model": tag,
            "Backbone": res["backbone"],
            "Best Epoch": res["best_epoch"],
            "Dev F1": res["best_dev_f1"],
            "Test Acc": res["test_metrics"]["accuracy"],
            "Test Macro F1": res["test_metrics"]["macro_f1"],
            "Per-class F1": str(res["test_metrics"]["per_f1"]),
        })
    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False))

    # Save summary
    summary_path = CONFIG["paths"]["checkpoint_root"] / "plm_comparison_summary.json"
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSummary saved: {summary_path}")

    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(summary_df))
    width = 0.35
    ax.bar([i - width/2 for i in x], summary_df["Dev F1"], width, label="Dev F1", color="steelblue")
    ax.bar([i + width/2 for i in x], summary_df["Test Macro F1"], width, label="Test F1", color="coral")
    ax.set_xticks(list(x))
    ax.set_xticklabels(summary_df["Model"], rotation=15)
    ax.set_ylabel("Macro F1")
    ax.set_title("PLM Comparison on ViFactCheck")
    ax.legend()
    plt.tight_layout()
    plt.savefig(CONFIG["paths"]["checkpoint_root"] / "plm_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()